In [1]:
cell_str=r'''
#include <stdio.h>
#include <stdlib.h>
#include <time.h>
#include <cuda_runtime.h>
#include <cublas_v2.h>

int main(int argc, char **argv) {
    if (argc < 2) {
        fprintf(stderr, "Uso: %s \n", argv[0]);
        return 1;
    }

    int n = atoi(argv[1]);
    size_t bytes = n * n * sizeof(float);

    float *h_a = (float*)malloc(bytes);
    float *h_b = (float*)malloc(bytes);
    float *h_c = (float*)malloc(bytes);

    for (int i = 0; i < n; i++) {
        for (int j = 0; j < n; j++) {
            h_a[i * n + j] = 2.0f;
            h_b[i * n + j] = 3.0f;
            h_c[i * n + j] = 0.0f;
        }
    }

    float *d_a, *d_b, *d_c;
    cudaMalloc(&d_a, bytes);
    cudaMalloc(&d_b, bytes);
    cudaMalloc(&d_c, bytes);

    cudaMemcpy(d_a, h_a, bytes, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, h_b, bytes, cudaMemcpyHostToDevice);

    // ---- CuBLAS Logic ----

    cublasHandle_t handle;
    cublasCreate(&handle);

    // As for the previous library implentations, we set alpha and beta respectively to 1.0 and 0.0 in order to compute C = A * B
    float alpha = 1.0f;
    float beta = 0.0f;

    // CUDA Timing events
    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    cudaEventRecord(start);

    // Launch of the function "Sgemm" (Single precision GEneral Matrix Multiply)
    cublasSgemm(handle, CUBLAS_OP_N, CUBLAS_OP_N,
                n, n, n,
                &alpha,
                d_b, n,
                d_a, n,
                &beta,
                d_c, n);

    cudaEventRecord(stop);
    cudaEventSynchronize(stop);

    float milliseconds = 0;
    cudaEventElapsedTime(&milliseconds, start, stop);
    printf("cuBLAS Time for n=%d: %.4f seconds\n", n, milliseconds / 1000.0f);

    cublasDestroy(handle);

    // ---- End ----


    cudaMemcpy(h_c, d_c, bytes, cudaMemcpyDeviceToHost);

    // Stampa di controllo
    FILE *f = fopen("mat-res.txt", "w");
    if (f) {
        fprintf(f, "%d\n\n", n);
        int sample = (n < 100) ? n : 10;
        for (int i = 0; i < sample; i++) {
            for (int j = 0; j < sample; j++) {
                fprintf(f, "%.0f ", h_c[i * n + j]);
            }
            fprintf(f, "\n");
        }
        fclose(f);
    }

    cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);
    free(h_a); free(h_b); free(h_c);

    return 0;
}
'''

with open('cuda_matrixmult_cuBLAS.cu', 'w') as f:
    f.write(cell_str)

In [2]:
!nvcc -arch=sm_75 -O3 -lcublas cuda_matrixmult_cuBLAS.cu -o cuda_matrixmult_cuBLAS

In [3]:
!nvprof ./cuda_matrixmult_cuBLAS 20000

==1807== NVPROF is profiling process 1807, command: ./cuda_matrixmult_cuBLAS 20000
cuBLAS Time for n=20000: 3.2406 seconds
==1807== Profiling application: ./cuda_matrixmult_cuBLAS 20000
==1807== Profiling result:
            Type  Time(%)      Time     Calls       Avg       Min       Max  Name
 GPU activities:   74.97%  3.16716s         1  3.16716s  3.16716s  3.16716s  volta_sgemm_128x128_nn
                   16.39%  692.51ms         2  346.26ms  340.40ms  352.11ms  [CUDA memcpy HtoD]
                    8.63%  364.67ms         1  364.67ms  364.67ms  364.67ms  [CUDA memcpy DtoH]
      API calls:   70.58%  3.16716s         1  3.16716s  3.16716s  3.16716s  cudaEventSynchronize
                   23.60%  1.05889s         3  352.96ms  340.63ms  365.09ms  cudaMemcpy
                    4.50%  201.80ms         6  33.634ms  17.586us  201.11ms  cudaMalloc
                    0.53%  24.001ms         1  24.001ms  24.001ms  24.001ms  cudaGetSymbolAddress
                    0.29%  12.946ms      